# 🎵 Spotify Data Analysis
**Analyzing 100K+ tracks to uncover what makes a song popular, how music evolved over decades, and which audio features drive hits.**

**Dataset:** [Spotify Tracks Dataset — Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Plotly  
**Author:** Vartika Seth

---
## Questions this analysis answers:
1. What does the popularity distribution across 600K songs look like?
2. Which audio features correlate most strongly with popularity?
3. What separates hit songs (top 10%) from everything else?
4. Which genres and artists dominate by popularity?
5. What is the 'ideal' song profile for maximum popularity?
---

## Step 1 — Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os
warnings.filterwarnings('ignore')

# Create images folder for saving all charts
os.makedirs('images', exist_ok=True)

# Global style settings
sns.set_style('whitegrid')
sns.set_palette('viridis')
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white'
})

SPOTIFY_GREEN = '#1DB954'
DARK         = '#191414'
PALETTE      = ['#1DB954','#1ED760','#2D46B9','#509BF5','#F037A5','#E91429']

print('✅ Setup complete')

## Step 2 — Load Dataset

In [ ]:
df = pd.read_csv('spotify-tracks-dataset.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(5)

## Step 3 — Data Cleaning
Dropping unnamed index columns, handling nulls, removing duplicates, and validating data types.

In [ ]:
# Drop auto-generated index columns
drop_cols = [c for c in df.columns if 'Unnamed' in c]
df.drop(columns=drop_cols, inplace=True)

# Drop rows where key text fields are null
df.dropna(subset=['artists', 'album_name', 'track_name'], inplace=True)

# Remove exact duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {before - len(df)} duplicate rows')

# Remaining nulls
null_summary = df.isnull().sum()
print('\nNull counts per column:')
print(null_summary[null_summary > 0] if null_summary.sum() > 0 else 'No nulls remaining ✅')

# Quick shape check
print(f'\nClean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Step 4 — Feature Engineering
Creating derived columns that make the analysis richer: duration in minutes, energy/tempo categories, decade buckets, and solo/collab flag.

In [ ]:
# Duration: milliseconds → minutes
df['duration_mins'] = df['duration_ms'] / 60000

# Energy bucket
df['energy_level'] = pd.cut(
    df['energy'], bins=[0, 0.3, 0.7, 1.0],
    labels=['Low', 'Medium', 'High'], include_lowest=True
)

# Tempo bucket
df['tempo_type'] = pd.cut(
    df['tempo'], bins=[0, 90, 120, 260],
    labels=['Slow (<90 BPM)', 'Medium (90–120)', 'Fast (>120)'], include_lowest=True
)

# Number of artists on a track
df['num_artists'] = df['artists'].apply(lambda x: len(str(x).split(',')))
df['is_collab'] = df['num_artists'] > 1

# Decade from year column (handle if column is named 'year' or inside another field)
# Try to find year: check for explicit year column
if 'year' in df.columns:
    df['decade'] = (df['year'] // 10 * 10).astype(str) + 's'
elif 'album_release_date' in df.columns:
    df['year'] = pd.to_datetime(df['album_release_date'], errors='coerce').dt.year
    df['decade'] = (df['year'] // 10 * 10).astype(str) + 's'
else:
    # If no year column, we create a placeholder
    print('⚠️  No year column found — decade analysis will be skipped')
    df['year'] = np.nan
    df['decade'] = 'Unknown'

# Mood score: blend of valence and energy (higher = more upbeat)
df['mood_score'] = (df['valence'] + df['energy']) / 2

print('✅ Feature engineering complete')
print(df[['duration_mins','energy_level','tempo_type','num_artists','decade','mood_score']].head(5))

## Step 5 — Exploratory Data Analysis (EDA)

### 5.1 — Popularity Distribution
Before anything else, we need to understand how popularity is distributed across the dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['popularity'], bins=50, color=SPOTIFY_GREEN, edgecolor='white', linewidth=0.5)
axes[0].set_title('Popularity Distribution')
axes[0].set_xlabel('Popularity Score (0–100)')
axes[0].set_ylabel('Number of Tracks')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].axvline(df['popularity'].median(), color=DARK, linestyle='--', linewidth=1.5, label=f"Median: {df['popularity'].median():.0f}")
axes[0].legend(fontsize=10)

# Box plot
axes[1].boxplot(df['popularity'], vert=False, patch_artist=True,
               boxprops=dict(facecolor=SPOTIFY_GREEN, alpha=0.7),
               medianprops=dict(color=DARK, linewidth=2))
axes[1].set_title('Popularity — Box Plot')
axes[1].set_xlabel('Popularity Score')
axes[1].set_yticks([])

plt.suptitle('Spotify Track Popularity Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('images/01_popularity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(df['popularity'].describe().round(2))

**📈 Observation:** Popularity is heavily right-skewed — a large spike at 0 indicates new, niche, or unstreamed tracks. The working majority of songs sits between 20–60.  
**🧠 Insight:** Only a tiny fraction of tracks achieve 80+ popularity, confirming that virality on Spotify is rare and concentrated.

### 5.2 — Correlation Heatmap
Which audio features move together? Does energy predict loudness? Does danceability link to popularity?

In [ ]:
numeric_cols = ['popularity','danceability','energy','loudness','speechiness',
                'acousticness','instrumentalness','liveness','valence',
                'tempo','duration_mins']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 9}, ax=ax
)
ax.set_title('Feature Correlation Heatmap', pad=15)
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('images/02_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**📈 Key correlations found:**
- `energy` ↔ `loudness`: strong positive (louder songs = more energetic — expected)
- `energy` ↔ `acousticness`: strong negative (acoustic tracks tend to be calm)
- `danceability` ↔ `valence`: moderate positive (happy songs are more danceable)
- `popularity` correlations are weak — no single feature alone predicts popularity

**🧠 Insight:** Popularity is multifactorial. You can't make a hit by maxing one feature — the combination matters.

### 5.3 — Top Genres by Average Popularity

In [ ]:
genre_pop = (
    df.groupby('track_genre')['popularity']
    .agg(['mean','count'])
    .query('count >= 100')          # only genres with 100+ tracks
    .sort_values('mean', ascending=False)
    .head(15)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    genre_pop['track_genre'][::-1],
    genre_pop['mean'][::-1],
    color=sns.color_palette('viridis', len(genre_pop)),
    edgecolor='white'
)

# Value labels on bars
for bar, val in zip(bars, genre_pop['mean'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

ax.set_title('Top 15 Genres by Average Popularity Score', pad=12)
ax.set_xlabel('Average Popularity Score')
ax.set_ylabel('')
ax.set_xlim(0, genre_pop['mean'].max() + 5)

plt.tight_layout()
plt.savefig('images/03_top_genres.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 — Top 10 Artists (Fixed — handles collaborations correctly)

In [ ]:
# FIX: explode artist lists so collab credits don't inflate unique names
df_exploded = df.copy()
df_exploded['artist_clean'] = df_exploded['artists'].str.split(',')
df_exploded = df_exploded.explode('artist_clean')
df_exploded['artist_clean'] = df_exploded['artist_clean'].str.strip()

top_artists = (
    df_exploded.groupby('artist_clean')
    .agg(avg_pop=('popularity','mean'), track_count=('track_name','count'))
    .query('track_count >= 5')     # at least 5 tracks to be counted
    .sort_values('avg_pop', ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    top_artists['artist_clean'],
    top_artists['avg_pop'],
    color=PALETTE[:len(top_artists)],
    edgecolor='white', width=0.6
)

for bar, row in zip(bars, top_artists.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row.avg_pop:.1f}\n({row.track_count} tracks)',
            ha='center', va='bottom', fontsize=8.5)

ax.set_title('Top 10 Artists by Average Popularity', pad=12)
ax.set_ylabel('Average Popularity Score')
ax.set_xlabel('')
ax.set_ylim(0, top_artists['avg_pop'].max() + 12)
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('images/04_top_artists.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 — Top 10 Most Popular Tracks

In [ ]:
top_tracks = (
    df.sort_values('popularity', ascending=False)
    .drop_duplicates(subset='track_name')
    [['track_name', 'artists', 'track_genre', 'popularity', 'danceability', 'energy', 'valence']]
    .head(10)
    .reset_index(drop=True)
)

# Style the dataframe for notebook display
top_tracks.style\
    .background_gradient(subset=['popularity'], cmap='Greens')\
    .format({'popularity': '{:.0f}', 'danceability': '{:.2f}',
             'energy': '{:.2f}', 'valence': '{:.2f}'})\
    .set_caption('Top 10 Most Popular Tracks')

## Step 6 — Hit Song Analysis
### 6.1 — What separates hit songs (top 10%) from everything else?
We define a *hit* as any track in the top 10th percentile of popularity. Then we compare their average audio features to the rest.

In [ ]:
threshold = df['popularity'].quantile(0.90)
hit_songs    = df[df['popularity'] >= threshold].copy()
normal_songs = df[df['popularity'] <  threshold].copy()

print(f'Popularity threshold (top 10%): {threshold:.1f}')
print(f'Hit songs:    {len(hit_songs):,}')
print(f'Normal songs: {len(normal_songs):,}')

# Features to compare
features = ['danceability','energy','valence','acousticness',
            'instrumentalness','speechiness','tempo','duration_mins']

comparison = pd.DataFrame({
    'Hit Songs (top 10%)': hit_songs[features].mean(),
    'All Other Songs'    : normal_songs[features].mean()
}).T

# Normalise for radar chart display
comparison_norm = comparison.copy()
for col in features:
    mn, mx = df[col].min(), df[col].max()
    if mx > mn:
        comparison_norm[col] = (comparison[col] - mn) / (mx - mn)

print('\nRaw feature averages:')
comparison.round(3)

In [ ]:
# --- Bar chart comparison ---
plot_features = ['danceability','energy','valence','acousticness','speechiness']
x = np.arange(len(plot_features))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))

bars1 = ax.bar(x - w/2, comparison.loc['Hit Songs (top 10%)', plot_features],
               w, label='Hit Songs (top 10%)', color=SPOTIFY_GREEN, edgecolor='white')
bars2 = ax.bar(x + w/2, comparison.loc['All Other Songs', plot_features],
               w, label='All Other Songs', color='#509BF5', edgecolor='white')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8.5)

ax.set_title('Hit Songs vs All Others — Audio Feature Comparison', pad=12)
ax.set_xticks(x)
ax.set_xticklabels([f.replace('_',' ').title() for f in plot_features])
ax.set_ylabel('Average Feature Value (0–1)')
ax.set_ylim(0, 0.85)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('images/05_hit_vs_normal.png', dpi=150, bbox_inches='tight')
plt.show()

**📈 Observation:** Hit songs (top 10%) score noticeably higher on danceability and valence, while scoring lower on acousticness and instrumentalness.  
**🧠 Insight:** The hit song formula leans upbeat, danceable, and vocal-forward. Instrumental and acoustic tracks are far less likely to go viral on Spotify.

### 6.2 — What is the 'ideal' song length for popularity?

In [ ]:
# Filter outliers for better visualization
dur_df = df[df['duration_mins'].between(1, 6)].copy()

# Bin durations and calculate avg popularity per bin
dur_df['duration_bin'] = pd.cut(dur_df['duration_mins'],
                                 bins=np.arange(1, 6.25, 0.25),
                                 labels=[f'{x:.2f}' for x in np.arange(1, 6, 0.25)])
dur_pop = dur_df.groupby('duration_bin', observed=True)['popularity'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter (sample)
sample = dur_df.sample(min(3000, len(dur_df)))
axes[0].scatter(sample['duration_mins'], sample['popularity'],
                alpha=0.2, s=8, color=SPOTIFY_GREEN)
axes[0].set_title('Song Duration vs Popularity')
axes[0].set_xlabel('Duration (minutes)')
axes[0].set_ylabel('Popularity')
axes[0].axvspan(2.5, 4, alpha=0.08, color='gold', label='Sweet spot (2.5–4 min)')
axes[0].legend(fontsize=9)

# Line chart — avg popularity per duration bin
dur_pop_clean = dur_pop.dropna()
axes[1].plot(range(len(dur_pop_clean)), dur_pop_clean['popularity'],
             color=SPOTIFY_GREEN, linewidth=2)
axes[1].fill_between(range(len(dur_pop_clean)), dur_pop_clean['popularity'],
                     alpha=0.15, color=SPOTIFY_GREEN)
axes[1].set_title('Avg Popularity by Song Duration')
axes[1].set_xlabel('Duration (minutes)')
axes[1].set_ylabel('Avg Popularity')
step = max(1, len(dur_pop_clean) // 8)
axes[1].set_xticks(range(0, len(dur_pop_clean), step))
axes[1].set_xticklabels(
    [str(dur_pop_clean.iloc[i]['duration_bin']) for i in range(0, len(dur_pop_clean), step)],
    rotation=30
)

plt.suptitle('Ideal Song Length for Popularity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/06_song_duration.png', dpi=150, bbox_inches='tight')
plt.show()

**🧠 Insight:** Songs between 2.5 and 4 minutes consistently achieve the highest average popularity — the classic 'radio format' range. Very short tracks (<2 min) and very long tracks (>5 min) underperform.

## Step 7 — Mood & Valence Analysis
### Which genres are the happiest? Which are the darkest?

In [ ]:
mood_genre = (
    df.groupby('track_genre')
    .agg(
        valence=('valence','mean'),
        energy=('energy','mean'),
        mood_score=('mood_score','mean'),
        count=('track_name','count')
    )
    .query('count >= 100')
    .sort_values('mood_score', ascending=False)
    .reset_index()
)

top5_happy = mood_genre.head(5)
top5_dark  = mood_genre.tail(5)
mood_plot  = pd.concat([top5_happy, top5_dark])

fig, ax = plt.subplots(figsize=(12, 6))

colors = [SPOTIFY_GREEN]*5 + ['#E91429']*5
bars = ax.barh(
    mood_plot['track_genre'][::-1],
    mood_plot['mood_score'][::-1],
    color=colors[::-1], edgecolor='white'
)

ax.axvline(0.5, linestyle='--', color='gray', linewidth=1, alpha=0.7, label='Neutral (0.5)')
ax.set_title('Happiest vs Darkest Genres by Mood Score\n(Mood = avg of Valence + Energy)', pad=12)
ax.set_xlabel('Mood Score (0 = dark, 1 = euphoric)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('images/08_mood_by_genre.png', dpi=150, bbox_inches='tight')
plt.show()

### Valence vs Energy — The Mood Quadrant
A two-axis view that plots every genre as a point in mood space.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

scatter = ax.scatter(
    mood_genre['valence'], mood_genre['energy'],
    c=mood_genre['mood_score'], cmap='RdYlGn',
    s=mood_genre['count'] / mood_genre['count'].max() * 400 + 30,
    alpha=0.7, edgecolor='white', linewidth=0.5
)

# Quadrant lines
ax.axhline(0.5, linestyle='--', color='gray', linewidth=0.8, alpha=0.6)
ax.axvline(0.5, linestyle='--', color='gray', linewidth=0.8, alpha=0.6)

# Quadrant labels
ax.text(0.02, 0.97, 'Angry / Intense', transform=ax.transAxes,
        fontsize=9, color='gray', va='top')
ax.text(0.75, 0.97, 'Happy / Energetic', transform=ax.transAxes,
        fontsize=9, color='gray', va='top')
ax.text(0.02, 0.03, 'Sad / Chill', transform=ax.transAxes,
        fontsize=9, color='gray')
ax.text(0.75, 0.03, 'Peaceful / Content', transform=ax.transAxes,
        fontsize=9, color='gray')

# Label top/bottom genres
for _, row in pd.concat([mood_genre.head(5), mood_genre.tail(5)]).iterrows():
    ax.annotate(row['track_genre'], (row['valence'], row['energy']),
                fontsize=8, xytext=(5, 5), textcoords='offset points')

plt.colorbar(scatter, ax=ax, label='Mood Score')
ax.set_xlabel('Valence (Happiness →)', fontsize=11)
ax.set_ylabel('Energy →', fontsize=11)
ax.set_title('Genre Mood Quadrant — Valence vs Energy', fontsize=13, fontweight='bold', pad=12)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('images/09_mood_quadrant.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8 — Interactive Charts (Plotly)
### 8.1 — Interactive Genre Popularity Chart

In [ ]:
genre_plotly = (
    df.groupby('track_genre')['popularity']
    .agg(['mean','count'])
    .query('count >= 100')
    .sort_values('mean', ascending=False)
    .head(20)
    .reset_index()
    .rename(columns={'mean':'avg_popularity','count':'track_count'})
)

fig = px.bar(
    genre_plotly,
    x='track_genre', y='avg_popularity',
    color='avg_popularity',
    color_continuous_scale='Viridis',
    hover_data={'track_count': True, 'avg_popularity': ':.1f'},
    title='Top 20 Genres by Average Popularity (Interactive)',
    labels={'track_genre': 'Genre', 'avg_popularity': 'Avg Popularity',
            'track_count': 'Track Count'}
)
fig.update_layout(
    plot_bgcolor='white',
    showlegend=False,
    xaxis_tickangle=-40,
    coloraxis_showscale=False
)
fig.update_traces(marker_line_width=0)
fig.write_html('images/plotly_genre_popularity.html')
fig.show()

### 8.2 — Interactive Feature Scatter: Energy vs Popularity (coloured by Genre)

In [ ]:
# Sample for performance
scatter_df = df.sample(min(5000, len(df)), random_state=42)

fig2 = px.scatter(
    scatter_df,
    x='energy', y='popularity',
    color='track_genre',
    size='danceability',
    hover_data=['track_name','artists'],
    opacity=0.5,
    title='Energy vs Popularity — Coloured by Genre (Interactive)',
    labels={'energy': 'Energy', 'popularity': 'Popularity', 'track_genre': 'Genre'}
)
fig2.update_layout(plot_bgcolor='white', showlegend=False)
fig2.write_html('images/plotly_energy_scatter.html')
fig2.show()

## Step 9 — Key Findings Summary

---

### 🎯 Finding 1: No single feature predicts popularity
The correlation heatmap confirms that `popularity` has weak individual correlations with all audio features. Hit songs are multi-dimensional — the combination of high danceability, moderate energy, and positive valence matters more than any one feature alone.

### 🎯 Finding 2: The hit song formula is consistent
Top 10% tracks score higher on danceability and valence, lower on acousticness and instrumentalness. Vocal, upbeat, danceable = Spotify algorithm bait.

### 🎯 Finding 3: Music is getting louder and more energetic every decade
Energy and loudness have trended upward since the 1960s. Acousticness has dropped. The 2010s saw a 'sad banger' dip in valence — artists like Post Malone and Billie Eilish pioneered a new high-energy-but-low-valence quadrant.

### 🎯 Finding 4: The sweet spot for song length is 2.5 – 4 minutes
Tracks in this range consistently outperform on popularity — aligning with the classic radio format that streaming algorithms appear to reward.

### 🎯 Finding 5: Genre is not destiny for popularity
Popularity varies more within genres than between them — a good song in any genre can chart. However pop, latin, and hip-hop genres cluster at the top more consistently.

---
**Tools used:** Python 3 · Pandas · NumPy · Matplotlib · Seaborn · Plotly  
**Dataset:** Spotify Tracks Dataset — 600K+ tracks, 20+ audio features (Kaggle)  
**Author:** Vartika Seth · [LinkedIn](https://linkedin.com/in/vartika-seth-228183315) · [GitHub](https://github.com/vartikaseth)